In [2]:
sam1=SAM()
sam1.load_data('SAM_CJ_joined_v2_cleaned_03122025.h5ad')

In [1]:
!pip install anndata==0.8.0

In [2]:
!pip install loompy

In [1]:
from samalg import SAM
import scanpy as sc
import statistics
import matplotlib.pyplot as plt
import seaborn as sns
import random
import pandas as pd
import matplotlib.colors
import scipy
import numpy as np
import sklearn.metrics as metrics
from scipy import sparse
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import loompy

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
sam1.adata.obs.columns

Index(['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'n_genes', 'n_counts',
       'key', 'hicat_merged', 'subclass_id_label_mapping',
       'subclass_id_label_lc', 'leiden_clusters',
       'subclass_id_label_mapping_nounlabeled', 'neurotransmitter',
       'region_label', 'subclass_id_label_reduced_mapping',
       'subclass_id_label_reduced_lc',
       'subclass_id_label_reduced_mapping_nounlabeled', 'fraction_match',
       'best_match', 'frac_match_test', 'leiden_removal', 'nCount_SCT',
       'nFeature_SCT', 'SCT_snn_res.0.8', 'seurat_clusters', 'SCT_snn_res.5',
       'eq_subclass', 'eq_subclass_lc', 'eq_subclass_frac',
       'eq_subclass_nounlabeled', 'eq_subclass_nounlabeled_NN',
       'eq_subclass_nounlabeled_nmm', 'ss_subclass', 'ss_subclass_nounlabeled',
       'ss_class', 'ss_subclass_nounlabeled_astro', 'ss_subclass_v2',
       'ss_subclass_v2_nounlabeled', 'ss_subclass_nounlabeled_nmm',
       'ss_subclass_v3_nounlabeled', 'subclass_id_label_crossed',
       'ss_subclas

In [4]:
sam1.adata.obs = sam1.adata.obs.drop(['hicat_merged', 'subclass_id_label_mapping',
       'subclass_id_label_lc', 'leiden_clusters',
       'subclass_id_label_mapping_nounlabeled', 'neurotransmitter',
       'region_label', 'subclass_id_label_reduced_mapping',
       'subclass_id_label_reduced_lc',
       'subclass_id_label_reduced_mapping_nounlabeled', 'fraction_match',
       'best_match', 'frac_match_test', 'leiden_removal', 'nCount_SCT',
       'nFeature_SCT', 'SCT_snn_res.0.8', 'seurat_clusters', 'SCT_snn_res.5',
       'eq_subclass', 'eq_subclass_lc', 'eq_subclass_frac',
       'eq_subclass_nounlabeled', 'eq_subclass_nounlabeled_NN',
       'eq_subclass_nounlabeled_nmm', 'ss_subclass', 'ss_subclass_nounlabeled',
       'ss_class', 'ss_subclass_nounlabeled_astro', 'ss_subclass_v2',
       'ss_subclass_v2_nounlabeled', 'ss_subclass_nounlabeled_nmm',
       'ss_subclass_v3_nounlabeled', 'subclass_id_label_crossed',
       'ss_subclass_nounlabeled_nmm_v2', 'subclass_id_label_crossed_NN',
       'subclass_id_label_crossed_nmm_v2_nn',
       'ss_subclass_nounlabeled_nmm_v2_nn',
       'ss_subclass_nounlabeled_nmm_v3_nn', 'ss_subclass_crossed_nmm_v3_nn',
       'ss_subclass_crossed_nn', 'neurotransmitter_v2',
       'ss_subclass_v4_nounlabeled', 'ss_subclass_v4_nounlabeled_nn',
       'ss_subclass_nounlabeled_nmm_v4_nn',
       'ss_subclass_nounlabeled_nmm_v4_nn_thresh30',
       'ss_subclass_nounlabeled_nmm_cl_v4_nn', 'neurotransmitter_v3'],axis = 1)

In [5]:
for i in range(0,30):
    dat = sc.read_loom('subset_Allen_institute_Full_subclass_test_250_'+str(i)+'.loom')
    dat.obs_names = dat.obs['obs_names']
    dat.var_names = list(dat.var['x'])
    
    sam=SAM(dat)
    sam.preprocess_data()
    sam.run()
    
    sam.adata.obs_names_make_unique()
    sam.adata.var_names_make_unique()

    sam1.adata.obs_names_make_unique()
    sam1.adata.var_names_make_unique()
    
    sams = {'mg':sam,'cj':sam1}

    sm = SAMAP(
        sams,
        f_maps = 'active_maps/hypo_proj/',
    )
    
    sm.run(pairwise=True)
    
    save_samap(sm , 'sm_Allen_Full_cj_v2_cleaned_08272026_subclass_250_'+str(i)+'.pkl')

/scratch/miniconda/lib/python3.7/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


RUNNING SAM
Iteration: 0, Convergence: 1.0
Iteration: 1, Convergence: 0.814847447618938
Computing the UMAP embedding...
Elapsed time: 368.1524176597595 seconds
Not updating the manifold...
Not updating the manifold...
18945 `mg` gene symbols match between the datasets and the BLAST graph.
14848 `cj` gene symbols match between the datasets and the BLAST graph.
Prepping datasets for translation.
Translating feature spaces pairwise.
Projecting data into joint latent space. 205.53860712051392
Correcting data with means. 269.08862376213074
Expanding neighbourhoods of species mg...
Expanding neighbourhoods of species cj...
Indegree coarsening
0/8 (0, 152957)
1/8 (20000, 152957)
2/8 (40000, 152957)
3/8 (60000, 152957)
4/8 (80000, 152957)
5/8 (100000, 152957)
6/8 (120000, 152957)
7/8 (140000, 152957)
Rescaling edge weights by expression correlations.
Concatenating SAM objects...
ITERATION: 0 
Average alignment score (A.S.):  0.5462836042872079 
Max A.S. improvement: 0.9834744069624592 
Min A.S

/scratch/miniconda/lib/python3.7/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


RUNNING SAM
Iteration: 0, Convergence: 1.0
Iteration: 1, Convergence: 0.814737514111529
Computing the UMAP embedding...
Elapsed time: 429.9770495891571 seconds
Not updating the manifold...
18945 `mg` gene symbols match between the datasets and the BLAST graph.
14848 `cj` gene symbols match between the datasets and the BLAST graph.
Prepping datasets for translation.
Translating feature spaces pairwise.
Projecting data into joint latent space. 206.7865858078003
Correcting data with means. 329.3349781036377
Expanding neighbourhoods of species mg...
Expanding neighbourhoods of species cj...
Indegree coarsening
0/8 (0, 152962)
1/8 (20000, 152962)
2/8 (40000, 152962)
3/8 (60000, 152962)
4/8 (80000, 152962)
5/8 (100000, 152962)
6/8 (120000, 152962)
7/8 (140000, 152962)
Rescaling edge weights by expression correlations.
Concatenating SAM objects...
ITERATION: 0 
Average alignment score (A.S.):  0.5470856252909707 
Max A.S. improvement: 0.98017875113188 
Min A.S. improvement: 0.0
Calculating ge

/scratch/miniconda/lib/python3.7/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


...

RUNNING SAM
Iteration: 0, Convergence: 1.0
Iteration: 1, Convergence: 0.8149469773001469
Computing the UMAP embedding...
Elapsed time: 277.73096919059753 seconds
Not updating the manifold...
18945 `mg` gene symbols match between the datasets and the BLAST graph.
14848 `cj` gene symbols match between the datasets and the BLAST graph.
Prepping datasets for translation.
Translating feature spaces pairwise.
Projecting data into joint latent space. 168.455176115036
Correcting data with means. 194.11796021461487
Expanding neighbourhoods of species mg...
Expanding neighbourhoods of species cj...
Indegree coarsening
0/8 (0, 152952)
1/8 (20000, 152952)
2/8 (40000, 152952)
3/8 (60000, 152952)
4/8 (80000, 152952)
5/8 (100000, 152952)
6/8 (120000, 152952)
7/8 (140000, 152952)
Rescaling edge weights by expression correlations.
Concatenating SAM objects...
ITERATION: 0 
Average alignment score (A.S.):  0.549848280519297 
Max A.S. improvement: 0.9861260373258874 
Min A.S. improvement: 0.0
Calculating

/scratch/miniconda/lib/python3.7/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


RUNNING SAM
Iteration: 0, Convergence: 1.0
Iteration: 1, Convergence: 0.8149167099317987
Computing the UMAP embedding...
Elapsed time: 286.5238609313965 seconds
Not updating the manifold...
18945 `mg` gene symbols match between the datasets and the BLAST graph.
14848 `cj` gene symbols match between the datasets and the BLAST graph.
Prepping datasets for translation.
Translating feature spaces pairwise.
Projecting data into joint latent space. 167.77265810966492
Correcting data with means. 197.06682872772217
Expanding neighbourhoods of species mg...
Expanding neighbourhoods of species cj...
Indegree coarsening
0/8 (0, 152958)
1/8 (20000, 152958)
2/8 (40000, 152958)
3/8 (60000, 152958)
4/8 (80000, 152958)
5/8 (100000, 152958)
6/8 (120000, 152958)
7/8 (140000, 152958)
Rescaling edge weights by expression correlations.
Concatenating SAM objects...
ITERATION: 0 
Average alignment score (A.S.):  0.5494368644822923 
Max A.S. improvement: 0.9886857298822317 
Min A.S. improvement: 0.0
Calculati

/scratch/miniconda/lib/python3.7/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


RUNNING SAM
Iteration: 0, Convergence: 1.0
Iteration: 1, Convergence: 0.8147183113042501
Computing the UMAP embedding...
Elapsed time: 298.81428241729736 seconds
Not updating the manifold...
18945 `mg` gene symbols match between the datasets and the BLAST graph.
14848 `cj` gene symbols match between the datasets and the BLAST graph.
Prepping datasets for translation.
Translating feature spaces pairwise.
Projecting data into joint latent space. 168.31756925582886
Correcting data with means. 194.75372004508972
Expanding neighbourhoods of species mg...
Expanding neighbourhoods of species cj...
Indegree coarsening
0/8 (0, 152959)
1/8 (20000, 152959)
2/8 (40000, 152959)
3/8 (60000, 152959)
4/8 (80000, 152959)
5/8 (100000, 152959)
6/8 (120000, 152959)
7/8 (140000, 152959)
Rescaling edge weights by expression correlations.
Concatenating SAM objects...
ITERATION: 0 
Average alignment score (A.S.):  0.5423482341714084 
Max A.S. improvement: 0.9791214844967808 
Min A.S. improvement: 0.0
Calculat